# Setup & Installations

In [ ]:
!nvidia-smi

Thu May 21 15:57:39 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
import torch
torch.cuda.empty_cache()

In [4]:
!pip install -q git+https://github.com/huggingface/transformers.git

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [5]:
!pip install -q accelerate bitsandbytes sentencepiece langchain_core langchain_huggingface pypdf torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.8/343.8 kB 12.3 MB/s eta 0:00:00


In [6]:
from huggingface_hub import login
from google.colab import userdata

secret_label = "HF_TOKEN"


login(userdata.get(secret_label))

# Imports

In [8]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, BitsAndBytesConfig
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from typing import Optional, List
import json

from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda

## Schema Configuration

In [7]:
class TestScores(BaseModel):
    """Schema for standardized test scores"""
    ielts: Optional[float] = Field(None, description="IELTS band score (0-9)")
    toefl: Optional[int] = Field(None, description="TOEFL score (0-120)")
    duolingo: Optional[int] = Field(None, description="Duolingo score (10-160)")
    gre_verbal: Optional[int] = Field(None, description="GRE Verbal Reasoning score")
    gre_quant: Optional[int] = Field(None, description="GRE Quantitative Reasoning score")
    gre_awa: Optional[float] = Field(None, description="GRE Analytical Writing score")


class UserProfile(BaseModel):
    """The final cleaned profile of the scholarship applicant"""
    email: Optional[str] = Field(None, description="Applicant's email")
    university: Optional[str] = Field(None, description="University name")
    degree_level: Optional[str] = Field(None, description="Degree level (e.g., BSc, MSc)")
    domain: Optional[str] = Field(None, description="Field of study (e.g., Computer Science, Electrical Engineering)")
    experience_years: Optional[int] = Field(None, description="Number of years of relevant experience")
    gpa: Optional[float] = Field(None, description="Grade Point Average")
    gpa_scale: Optional[float] = Field(4.0, description="GPA scale, e.g., 4.0 or 5.0")
    test_scores: TestScores = Field(default_factory=TestScores)
    project_titles: List[str] = Field(default_factory=list, description="List of project titles")
    published_research_titles: List[str] = Field(default_factory=list, description="List of published research paper titles")
    competition_wins: List[str] = Field(default_factory=list, description="List of competitions won")
    volunteering_activities: List[str] = Field(default_factory=list, description="List of volunteering activities")
    willing_to_return: Optional[bool] = Field(None, description="Indicates whether the applicant is willing to return to their home country after graduation (Yes/No)")
    graduation_certificate: Optional[bool] = Field(None, description="Indicates whether the applicant has a graduation certificate (Yes/No)")


## Model Loading

In [ ]:
MODEL_NAME="google/gemma-2-9b-it"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config, # to prevent OOM
    device_map="auto"
)

print("-- Model Loaded --")

Loading weights:   0%|          | 0/464 [00:00<?, ?it/s]

-- Model Loaded --


In [ ]:
pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    temperature=0.7,
    do_sample=True,
    repetition_penalty=1.2,
    return_full_text=False
)

llm = ChatHuggingFace(llm=HuggingFacePipeline(pipeline=pipeline))

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature', 'do_sample', 'repetition_penalty'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


# Agent1 Data Extractor

## Chain Logic

In [ ]:
parser = PydanticOutputParser(pydantic_object=UserProfile)

SYSTEM_PROMPT = ChatPromptTemplate.from_messages([
    ("human", """You are an expert Admissions Profiler AI.

Your job is to extract structured information from a resume or personal statement and convert it into valid JSON.

========================
EXTRACTION RULES
========================

1. ONLY extract information explicitly stated in the text.
   - Do NOT infer, guess, or hallucinate anything.

2. If a field is missing:
   - use null for strings/numbers
   - use [] for lists

3. GPA:
   - extract GPA value AND scale if mentioned (default = 4.0 if not mentioned)

4. STRICT EXTRACTION:
   - Keep EXACT wording for:
     • projects
     • research papers
     • competitions
     • volunteering activities

5. NORMALIZATION (IMPORTANT):
   - If experience is mentioned (e.g. "2 years experience"):
     → extract as number of years (int or float if possible)

   - If graduation certificate is mentioned:
     → set graduation_certificate = yes

   - If user says they want to stay in country of scholarship:
     → extract as boolean field: willing_to_return = no

   - If a "field/domain/major area" is implied:
     → extract into "domain" (e.g. Computer Science, AI, etc.)

========================
OUTPUT FORMAT RULES
========================

{format_instructions}

- Respond ONLY with valid JSON
- No explanations
- No markdown
- No extra text

========================
INPUT TEXT
========================

{applicant_text}
""")
])

## Clean Json

In [ ]:
import re

In [ ]:
def clean_json_output(output_string: str) -> str:
    output_string = output_string.replace("```json", "").replace("```", "")
    start = output_string.find('{')
    end = output_string.rfind('}') + 1
    if start != -1 and end > start:
        return output_string[start:end]
    return output_string

In [ ]:
def extract_json(text: str) -> str:
    """Finds the first valid JSON object ({...}) in a string."""
    # Remove markdown code blocks if the LLM added them
    text = text.replace("```json", "").replace("```", "")

    # Find the first { and the last }
    match = re.search(r'\{.*\}', text, re.DOTALL)
    if match:
        return match.group(0)
    else:
        raise ValueError("No JSON object found in the LLM output.")

In [ ]:
chain1 = (
    SYSTEM_PROMPT.partial(
        format_instructions=parser.get_format_instructions()
    )
    | llm
    | StrOutputParser()
    | RunnableLambda(clean_json_output))

print("-- chain created successfully! --")

-- chain created successfully! --


## Extract text from PDF

In [ ]:
def extract_text_from_pdf(file_bytes) -> str:
    """Reads PDF bytes and returns cleaned text."""
    try:
        pdf_file = io.BytesIO(file_bytes)
        reader = PdfReader(pdf_file)
        raw_text = ""
        for page in reader.pages:
            raw_text += page.extract_text() + "\n"

        clean_text = re.sub(r'\s+', ' ', raw_text).strip()
        return clean_text
    except Exception as e:
        print(f"Error reading PDF: {e}")
        return ""

In [ ]:
def process_agent_input(input_type, text_data, file_bytes):
    """Core function to process either text or PDF and run the chain"""
    if input_type == 'Upload PDF':
        if not file_bytes:
            return "Please upload a PDF file first."
        text_data = extract_text_from_pdf(file_bytes)
        if not text_data:
            return "Could not extract text from PDF. Is it a scanned image?"

    if not text_data.strip():
        return "No data provided. Please type text or upload a PDF."

    # Truncate if text is too long for the model context
    if len(text_data) > 4000:
        text_data = text_data[:4000]

## Test text

In [ ]:
sample_text = """
Hi, my name is Abdallah.
I graduated from the University of Toronto with a BSc in Computer Science.
My GPA is 3.8 out of 4.0.
I took the TOEFL and scored 112.
Also did the GRE and got Verbal 162, Quant 170, AWA 4.5.
For projects, I built a "Real-time Sign Language Translator" and a "Decentralized Voting System using Blockchain".
I have two published papers: "Optimizing LSTM for Real-Time Video Processing" and "Blockchain Consensus Mechanisms in IoT Networks".
I won 1st place in the Google Hackathon 2023 and got a Bronze medal in the ICPC Regional Finals.
In my free time, I volunteer at Code.org teaching kids to code, and I also help out at the local animal shelter.
i need to get back my country after finishing scholarship
i have 4 years of expierence
i dont have a graduation certificate

"""

print("\nRunning Agent 1")
try:
    raw_output = chain1.invoke({"applicant_text": sample_text})
    print(raw_output)
    if hasattr(raw_output, "content"):
        profile = raw_output.content

    cleaned = extract_json(raw_output)

    data = json.loads(cleaned)

    profile = UserProfile(**data)

    print("\n-- EXTRACTION SUCCESSFUL! --")
    print("-" * 30)
    print(f"University:          {profile.university}")
    print(f"Domain:          {profile.domain}")
    print(f"Degree level:              {profile.degree_level}")
    print(f"Graduation Certificate:        {profile.graduation_certificate}")
    print(f"GPA:                 {profile.gpa}/{profile.gpa_scale}")
    print(f"TOEFL:               {profile.test_scores.toefl}")
    print(f"GRE (V/Q/AWA):       {profile.test_scores.gre_verbal} / {profile.test_scores.gre_quant} / {profile.test_scores.gre_awa}")
    print(f"Experiecnce Years:            {profile.experience_years}")
    print(f"Projects:            {profile.project_titles}")
    print(f"Research Titles:     {profile.published_research_titles}")
    print(f"Competitions:        {profile.competition_wins}")
    print(f"Volunteering:        {profile.volunteering_activities}")
    print(f"Willing to return:        {profile.willing_to_return}")

except Exception as e:
    print(f"\n-- EXTRACTION FAILED: {e} --")


Running Agent 1


[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GemmaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


{
  "email": null,
  "university": "University of Toronto",
  "degree_level": "BSc",
  "domain": "Computer Science",
  "experience_years": 4,
  "gpa": 3.8,
  "gpa_scale": 4.0,
  "test_scores": {
    "toefl": 112,
    "gre_verbal": 162,
    "gre_quant": 170,
    "gre_awa": 4.5,
    "ielts": null,
    "duolingo": null
  },
  "project_titles": [
    "Real-time Sign Language Translator",
    "Decentralized Voting System using Blockchain"
  ],
  "published_research_titles": [
    "Optimizing LSTM for Real-Time Video Processing",
    "Blockchain Consensus Mechanisms in IoT Networks"
  ],
  "competition_wins": [
    "1st place in the Google Hackathon 2023",
    "Bronze medal in the ICPC Regional Finals"
  ],
  "volunteering_activities": [
    "teaching kids to code at Code.org",
    "helping out at the local animal shelter"
  ],
  "willing_to_return": false,
  "graduation_certificate": false
}

-- EXTRACTION SUCCESSFUL! --
------------------------------
University:          University of Toro

In [ ]:
import torch
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

## Test pdf

In [ ]:
import ipywidgets as widgets
from IPython.display import display
from pypdf import PdfReader
import io
import re
import time

In [ ]:
print("Please upload a CV/Resume PDF:")
uploader = widgets.FileUpload(
    accept='.pdf',
    multiple=False  # Only one file at a time
)
display(uploader)

Please upload a CV/Resume PDF:


FileUpload(value=(), accept='.pdf', description='Upload')

In [ ]:
import time
import json

# Wait for upload
while not uploader.value:
    print("Waiting for file upload...")
    time.sleep(2)

print("File uploaded! Extracting text from PDF...")

# Get bytes
uploaded_file = uploader.value[0]
file_bytes = uploaded_file['content']

pdf_text = extract_text_from_pdf(file_bytes)

if pdf_text:
    # print(f"\nPreview (first 300 chars):\n{pdf_text[:300]}...\n")

    print("Running Agent 1: Asking LLM to extract data...")

    raw_llm_output = chain1.invoke({"applicant_text": pdf_text})

    print("Attempting to parse into Pydantic object...")
    try:
        profile = UserProfile.model_validate_json(raw_llm_output)

        print("\nEXTRACTION SUCCESSFUL FROM PDF!")
        print("-" * 30)
        print(f"University:          {profile.university}")
        print(f"Degree:              {profile.degree_and_specialty}")
        print(f"GPA:                 {profile.gpa}/{profile.gpa_scale}")
        print(f"TOEFL:               {profile.test_scores.toefl}")
        print(f"Projects:            {profile.project_titles}")

    except Exception as e:
        print(f"\nPARSING FAILED: {e}")
        print("The LLM did not return valid JSON that matches our schema.")
else:
    print("Could not extract text from the PDF.")

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


File uploaded! Extracting text from PDF...
Running Agent 1: Asking LLM to extract data...
Attempting to parse into Pydantic object...

EXTRACTION SUCCESSFUL FROM PDF!
------------------------------
University:          Faculty of Computers and Informatics, Suez Canal University
Degree:              Computer Science
GPA:                 3.2/4.0
TOEFL:               None
Projects:            ['Ai Portfolio', 'Chatbot', 'Face Recognition', 'Sentiment Analysis']
